In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import warnings
from tqdm import tqdm
import os
from transformers import AutoModel

from utils.tokenizersM import get_ntv3_tokenizer
import math
import torch.nn.functional as F
import random
warnings.filterwarnings('ignore')

# Ensure reproducibility: make results consistent across runs
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(42)

def _read_signal_file(file, m=False):
    signals = [] 
    with open(file) as f:

        for line in f:
            parts = line.strip().split()
            if m:
                signal_values = [x * 0.01 for x in map(float, parts[4:])]
            else:
                signal_values = list(map(float, parts[4:]))  # 从第5列开始读取200个值

            signals.append(signal_values)

    signals = torch.tensor(signals, dtype=torch.float32)
    # pad = torch.zeros(signals.size(0), 1)  # [N, 1]

    # signals = torch.cat([pad, signals, pad], dim=-1)
    return  signals  

        
class GenomicDataset(Dataset):
    def __init__(self, fasta_pos_fu, methylation_pos_fusame,  methylation_pos_fudif,  fasta_pos_z, methylation_pos_zsame,  methylation_pos_zdif, 
    fasta_neg_fu, methylation_neg_fusame, methylation_neg_fudif, fasta_neg_z, methylation_neg_zsame, methylation_neg_zdif,
    Dnase_Z_F,Dnase_Z_Z,Dnase_F_F,Dnase_F_Z ,myname
   
    ):
        self.fasta_pos = self.read_fasta(fasta_pos_fu) + self.read_fasta(fasta_pos_z) 
        self.fasta_neg = self.read_fasta(fasta_neg_fu) + self.read_fasta(fasta_neg_z) 
        sequences = self.fasta_pos + self.fasta_neg
 
        # 初始化 tokenizer
        tokenizer = get_ntv3_tokenizer()

        # tokenizer 输出 numpy array
        tokens_np = tokenizer.batch_np_tokenize(sequences)  # shape: [batch, seq_len]
        self.input_ids = torch.tensor(tokens_np, dtype=torch.long)

        # Dnase_Z_F = _read_signal_file(Dnase_Z_F)
        # Dnase_Z_Z = _read_signal_file(Dnase_Z_Z)
        # Dnase_F_F = _read_signal_file(Dnase_F_F)
        # Dnase_F_Z = _read_signal_file(Dnase_F_Z)
        # self.Dnase = np.concatenate([
        #                     torch.flip(Dnase_Z_F, dims=[1])  , 
        #                     Dnase_Z_Z , 
        #                     torch.flip(Dnase_F_F, dims=[1]) , 
        #                     Dnase_F_Z, 
        #                     ], axis=0)

        # self.labels = np.array([1] * len(self.fasta_pos) + [0] * len(self.fasta_neg))

        # # Save preprocessed features if not exist
        labels_path = f'L1k_1_k_{myname}_labels_S.pt'
        if os.path.exists(labels_path):
            print(f"{labels_path} already exists. Loading labels...")
            self.labels = torch.load(labels_path)#.numpy()
            # self.labels2 = torch.load(f'/home/hjzhang/dataset/home-1/ylxiong/Center/Hela_input/processed/Hela_labels.pt')#.numpy()
            # self.labels = torch.cat([ self.labels , self.labels2],0) # 添加首尾

        else:
            torch.save(torch.tensor(self.labels, dtype=torch.int32), labels_path)
            print(f"Saved labels to {labels_path}")

        # methylation_pos_fudif = _read_signal_file(methylation_pos_fusame  ,m=True)
        # methylation_pos_zdif = _read_signal_file(methylation_pos_zsame ,m=True)
        # methylation_neg_fudif = _read_signal_file(methylation_neg_fusame ,m=True)
        # methylation_neg_zdif = _read_signal_file(methylation_neg_zsame ,m=True)

        # self.methylation = np.concatenate([
        #                     torch.flip(methylation_pos_fudif, dims=[1])  , 
        #                     methylation_pos_zdif , 
        #                     torch.flip(methylation_neg_fudif, dims=[1]) , 
        #                     methylation_neg_zdif , 
        #                     ], axis=0)
             

        epi_path = f'L1k_1_k_{myname}_epis_WGBS_3_S.pt'
        if os.path.exists(epi_path):
            print(f"{epi_path} already exists. Loading compressed features...")
            self.epi = torch.load(epi_path).float() 
            print(self.epi.max())

        else:
            print("Computing and saving compressed epi features...")
            epi_tensor = torch.tensor( self.methylation[:, :, np.newaxis]  , dtype=torch.float32)
            self.epi = epi_tensor.float()
            print(f"Saved compressed epi features to {epi_path}")
            print(self.epi.max())
            torch.save(self.epi, epi_path)


        Dnase_path = f'L1k_1_k_{myname}_epis_dnase_S.pt'
        if os.path.exists(Dnase_path):
            print(f"{Dnase_path} already exists. Loading compressed features...")
            self.Dnase = torch.log1p(torch.load(Dnase_path).float())
            print(self.Dnase.max())
        else:
            print("Computing and saving compressed epi features...")
            epi_tensor = torch.tensor( self.Dnase[:, :, np.newaxis]  , dtype=torch.float32)
            torch.save(epi_tensor, Dnase_path)
            self.Dnase =  torch.log1p( epi_tensor.float())
            print(f"Saved compressed epi features to {Dnase_path}")

    def read_fasta(self, file):
        seqs = []
        with open(file) as f:
            seq = ''
            for line in f:
                if line.startswith('>'):
                    if seq:
                        seqs.append(seq.upper())
                        seq = ''
                else:
                    seq += line.strip()
            if seq:
                seqs.append(seq.upper())
        return seqs

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        train_seq = torch.cat([ self.Dnase[idx],  self.epi[idx]],-1) # 添加首尾

        return (
            self.input_ids[idx] , train_seq ,
            torch.tensor(self.labels[idx], dtype=torch.long),        idx  # 新增
        )


class AdaptiveLayerNorm(nn.LayerNorm):
    """LayerNorm that applies per-condition affine modulation."""

    def __init__(
        self, num_features: int, conditions_dims: list[int], epsilon: float = 1e-5
    ):
        super().__init__(
            normalized_shape=num_features, eps=epsilon, elementwise_affine=True
        )
        self.modulation_layers = nn.ModuleList(
            [nn.Linear(cd, 2 * num_features) for cd in conditions_dims]
        )
        self._num_conditions = len(conditions_dims)
        self._dim = num_features

    def forward(  # type: ignore[override]
        self,
        x: torch.Tensor,
        conditions: list[torch.Tensor],
        conditions_masks: list[torch.Tensor] | None = None,
    ) -> torch.Tensor:
        x = self._base_ln_fp32(x)

        if len(conditions) != self._num_conditions:
            raise ValueError("Number of conditions mismatch")

        if conditions_masks is None:
            conditions_masks = [
                torch.ones(x.shape[0], dtype=x.dtype, device=x.device)
                for _ in conditions
            ]

        scale = torch.ones_like(x[:, :1, :])
        shift = torch.zeros_like(x[:, :1, :])

        for i, cond in enumerate(conditions):
            cond_cast = cond.to(self.modulation_layers[i].weight.dtype)
            tmp = self.modulation_layers[i](cond_cast).unsqueeze(1)
            tmp = tmp.to(x.dtype)
            shift_i, scale_i = torch.chunk(tmp, 2, dim=-1)
            mask = conditions_masks[i].unsqueeze(-1).unsqueeze(-1)
            shift_i = torch.where(mask.bool(), shift_i, 0.0)
            scale_i = torch.where(mask.bool(), scale_i, 0.0)
            scale = scale * (1.0 + scale_i)
            shift = shift + shift_i

        return x * scale + shift

    def _base_ln_fp32(self, x: torch.Tensor) -> torch.Tensor:
        """Run base LayerNorm in fp32 (compiler-friendly, like Mistral/Gemma)."""
        # Compute in fp32
        x_fp32 = x.to(torch.float32)
        mean = x_fp32.mean(dim=-1, keepdim=True)
        var = ((x_fp32 - mean) ** 2).mean(dim=-1, keepdim=True)
        x_normed = (x_fp32 - mean) * torch.rsqrt(var + self.eps)
        
        # Apply inherited weight/bias in fp32, then cast back
        if self.weight is not None:
            x_normed = x_normed * self.weight.to(torch.float32)
        if self.bias is not None:
            x_normed = x_normed + self.bias.to(torch.float32)
        
        return x_normed.type_as(x)

class MultiHeadAttention(nn.Module):
    """
    A vanilla multi-head masked self-attention layer with a projection at the end.
    It is possible to use torch.nn.MultiheadAttention here but I am including an
    explicit implementation here to show that there is nothing too scary here.
    """

    def __init__(self, n_embd):
        super().__init__()
        n_head = 12
        block_size = 1024
        # key, query, value projections for all heads
        self.key = nn.Linear(n_embd, n_embd)
        self.query = nn.Linear(n_embd, n_embd)
        self.value = nn.Linear(n_embd, n_embd)
        # regularization
        self.attn_drop = nn.Dropout(0.1)
        self.resid_drop = nn.Dropout(0.1)
        # output projection
        self.proj = nn.Linear(n_embd, n_embd)
        # causal mask to ensure that attention is only applied to the left in the input sequence
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size))
                             .view(1, 1, block_size, block_size))

        self.n_head = n_head        

    def forward(self, q, k, v, layer_past=None, condition=None, mask=True):
        B, T, C = q.size()

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        k = self.key(k).view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = self.query(q).view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = self.value(v).view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))

        if mask:
            att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        attn_save = att
        att = self.attn_drop(att)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_drop(self.proj(y))
        return y, attn_save


/home/hjzhang/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
name_list = ["A549", # 1
            "K562",   # 2
            "HepG2", # 3
            "293T", # 4
            ]
name = name_list[0]
name2 = name_list[1]
name3 = name_list[2]
name4 = name_list[3]
all_attn = {}

batch_size = 1
device_ids = [ 0 ] 
if True:
    if True:
        dataset = GenomicDataset(   
                f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/G4_1k.suF.fa",
                f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/G4_1k.suF_F.tsv",
                f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/G4_1k.suF_Z.tsv",
                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/G4_1k.suZ.fa",
                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/G4_1k.suZ_Z.tsv",
                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/G4_1k.suZ_F.tsv",

                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/noG4_1k.suF.fa",
                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/noG4_1k.suF_F.tsv",
                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/noG4_1k.suF_Z.tsv",

                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/noG4_1k.suZ.fa",
                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/noG4_1k.suZ_Z.tsv",
                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/noG4_1k.suZ_F.tsv",

                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/G4_1k.suF_D.tsv",
                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/G4_1k.suZ_D.tsv",
                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/noG4_1k.suF_D.tsv",
                                        f"home/hjzhang/dataset/home-1/ylxiong/_update/Intervene_results/{name}_1k/noG4_1k.suZ_D.tsv", name
                )

L1k_1_k_A549_labels_S.pt already exists. Loading labels...
L1k_1_k_A549_epis_WGBS_3_S.pt already exists. Loading compressed features...
tensor(1.)
L1k_1_k_A549_epis_dnase_S.pt already exists. Loading compressed features...
tensor(8.9041)


In [ ]:

def _edit_distance_one_tokens(input_ids, start, end, alphabet_tokens=(6,7,8,9)):
    """Generate all sequences with single nucleotide substitutions.

    input_ids: tensor [1, L]
    Returns tensor [num_mutations, L]
    """

    seq = input_ids.clone()
    L = seq.shape[1]

    mutations = []

    for pos in range(start, end):
        original = seq[0, pos].item()

        for a in alphabet_tokens:
            mut = seq.clone()
            mut[0, pos] = a
            mutations.append(mut)

    return torch.cat(mutations, dim=0)


def saturation_mutagenesis_tokens(model, input_ids, species_ids,  w,
                                  start, end, batch_size=32, device='cuda'):
    """Run saturation mutagenesis for tokenized DNA sequence.

    Returns:
        importance matrix [alphabet, position]
    """

    model.eval()

    with torch.no_grad():

        # original prediction
        y0, _ = model(input_ids, species_ids, w)
        y0 = torch.softmax(y0, dim=1)[:,1]

        mutated_inputs = _edit_distance_one_tokens(input_ids, start, end)

        # repeat auxiliary inputs
        species_rep = species_ids.repeat(mutated_inputs.shape[0])
        w_rep = w.repeat(mutated_inputs.shape[0], 1, 1)

        preds = []

        for i in range(0, mutated_inputs.shape[0], batch_size):

            batch_ids = mutated_inputs[i:i+batch_size].to(device)
            batch_species = species_rep[i:i+batch_size].to(device)
            batch_w = w_rep[i:i+batch_size].to(device)

            out, _ = model(batch_ids, batch_species, batch_w)
            prob = torch.softmax(out, dim=1)[:,1]

            preds.append(prob.cpu())

        preds = torch.cat(preds)

    L = end - start

    preds = preds.view(L, 4)

    y0_val = y0.item()

    attr = preds - y0_val      # [L,4]
    attr = attr - attr.mean(dim=1, keepdim=True)

    importance = preds - y0_val

    return importance.T, attr.T

        
import math
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import itertools
import torch
from tangermeme.plot import plot_logo
import torch.nn as nn

# TF-MoDISco
import h5py

class Block(nn.Module):
    """ an unassuming Transformer block """

    def __init__(self, n_embd):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(0.1),
        )

    def forward(self, x, condition=None, mask=True):
        x = self.ln1(x)
        y, attn = self.attn(x, x, x, condition=condition, mask=mask)
        x = x + y
        x = x + self.mlp(self.ln2(x))
        return x, attn

class G4former(nn.Module):
    def __init__(self):
        super(G4former, self).__init__()     

        self.adaptive_norm = AdaptiveLayerNorm(
            num_features=768, 
            conditions_dims=[ 1024 ]  # w 的条件维度，例如2或更多
        )
        self.fc2 = nn.Linear(1, 768)  # Second fully connected layer (optional)
        self.hidden_dim = 768

        self.classifier = nn.Linear(self.hidden_dim, 2)  # Binary classification
        self.activation = nn.GELU()  # Activation function

        self.pytorch_model = AutoModel.from_pretrained(
                "/home/hjzhang/ntv3G4/nucleotide_transformer_v3/NTv3_100m_post",
                    trust_remote_code=True
                )

        self.center_len = 256
        self.Block = Block(self.hidden_dim)



    def plot_attention(self, attn, input_ids=None, save_path="attention_map.png"):
        """Plot attention heatmap with nucleotide labels"""

        token_dict = {
            0:"<unk>",1:"<pad>",2:"<mask>",3:"<cls>",4:"<eos>",
            5:"<bos>",6:"A",7:"T",8:"C",9:"G",10:"N"
        }

        # attn shape
        if attn.dim() == 4:
            attn = attn[0]        # first sample

        if attn.dim() == 3:
            attn = attn.mean(0)   # average heads

        attn = attn.detach().cpu().numpy()

        # 转换序列
        seq_labels = None
        if input_ids is not None:
            start_index = (input_ids.shape[1] - 50) // 2
            end_index = start_index + 50
            seq_tokens = input_ids[0, start_index:end_index].detach().cpu().numpy()
            seq_labels = [token_dict.get(int(i),"N") for i in seq_tokens]

        plt.figure(figsize=(8,8))
        # mask upper triangle so only lower triangle is visible
        mask = np.triu(np.ones_like(attn, dtype=bool), k=1)
        ax = sns.heatmap(attn, cmap="magma", mask=mask)

        if seq_labels is not None:
            L = min(len(seq_labels), attn.shape[0])
            ax.set_xticks(range(L))
            ax.set_yticks(range(L))
            ax.set_xticklabels(seq_labels[:L], rotation=0)
            ax.set_yticklabels(seq_labels[:L], rotation=0)

        plt.xlabel("Key")
        plt.ylabel("Query")
        plt.title("Attention Heatmap")

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()

    def forward(self,input_ids, species_ids, w ):
        # input_ids = input_ids.float).long()
        outs = self.pytorch_model(input_ids, species_ids=species_ids )
        x  = outs.embedding # torch.Size([2, 288, 7362]) torch.Size([64, 256, 768]) 11 + 2

        cond = w # shape [B, L, 2]

        res = self.activation( self.fc2(cond[..., 1].unsqueeze(-1))) * cond[..., 1].unsqueeze(-1)
        # 通过 fc2 把 w 映射到同维度
        x = x + res 

        conditions = [ cond[..., 0]]
        # 用 AdaptiveLayerNorm 做条件调制
        x = self.adaptive_norm(x, conditions=conditions)
        outputs, attn = self.Block(x)
        
        start_index = (outputs.shape[1] - 50) // 2
        end_index = start_index + 50
        x = outputs[:, start_index: end_index, :]
            
        attn = attn[:, :, start_index: end_index,start_index: end_index]
        
        logits = self.classifier(x.mean(1))
        return logits, attn
    
    def reset_parameters(self):
        def init_layer(m):
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        # 递归应用到所有子模块
        self.fc2.apply(init_layer)
        self.classifier.apply(init_layer)


batch_size = 1
device_ids = [ 0 ] 
criterion = nn.CrossEntropyLoss(  )
device = f"cuda:{device_ids[0]}"
if device:
    species_token = 27
    conut = 0
        
    val_accs, val_losses = [] ,[]
        # ---------------------- 验证集：同步收集embedding和标签y ----------------------
    all_attrs = []
    all_onehot = []
    if name == "293T":
        best_model_wts = f'save_L1k_-11-58_293T_64_3e-05_293T.pth'

    else:
        best_model_wts = f'save_L1k_-11-58_epoch_1_64_3e-05_{name}.pth'
        
    if best_model_wts:
        idx_save_path = f'L1k_1_k_{name}_split_indices.pt'
        if os.path.exists(idx_save_path):
            print(f"Loading existing split indices from {idx_save_path}")
            split_indices = torch.load(idx_save_path)
            train_idx = split_indices["train"]
            val_idx = split_indices["val"]
            test_idx = split_indices["test"]
            print( len(test_idx) )
        test_subset = torch.utils.data.Subset(dataset, test_idx)
        test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)
        model = G4former().to(device)
        # save_L1k_-11-58_epoch_1_64_{lr}_{name}.pth'
        if model:
            state_dict = torch.load(best_model_wts, map_location=device)            
            new_state_dict = {}
            for key, value in state_dict.items():
                # 移除开头的"module."
                new_key = key[7:] if key.startswith('module.') else key
                new_state_dict[new_key] = value

        model.load_state_dict(new_state_dict)
        model = torch.nn.DataParallel(model, device_ids=device_ids)
        model.eval()
        loop = tqdm(test_loader, desc=f"[Test]")

        if loop:
            val_loss, val_correct, val_total = 0, 0, 0
            val_preds, val_targets, val_probs = [], [], []
            with torch.no_grad():
                for input_ids, w, y, idx in loop:
                    if conut == 5:
                        break

                    input_ids,w, y = input_ids.to(device), w.to(device), y.to(device)

                    species_ids = torch.tensor([species_token] * len(y), dtype=torch.long).to(device)

                    out, attn = model(input_ids, species_ids, w)

                    # Save attention visualization for first batch
                    loss = criterion(out, y)
                    val_loss += loss.item() * y.size(0)

                    probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
                    pred_labels = out.argmax(1).cpu().numpy()

                    if y.cpu().numpy()[0] == pred_labels[0]:
                        if pred_labels[0] ==1:
                            if isinstance(model, torch.nn.DataParallel):
                                # model.module.plot_attention(attn, input_ids, f"/home/hjzhang/dataset/home-1/ylxiong/caduceus/ISM/{name}.{conut}.attention_map.pdf")

                                # ===== Saturation mutagenesis for first example =====
                                start_index = (input_ids.shape[1] - model.module.center_len) // 2 if isinstance(model, torch.nn.DataParallel) else (input_ids.shape[1] - model.center_len) // 2
                                end_index = start_index + (model.module.center_len if isinstance(model, torch.nn.DataParallel) else model.center_len)

                                if isinstance(model, torch.nn.DataParallel):
                                    ism_map, attr = saturation_mutagenesis_tokens(
                                        model.module,
                                        input_ids[:1],
                                        species_ids[:1],
                                        w[:1],
                                        start_index,
                                        end_index,
                                        device=device
                                    )
                                else:
                                    ism_map, attr = saturation_mutagenesis_tokens( # torch.Size([4, 50])
                                        model,
                                        input_ids[:1],
                                        species_ids[:1],
                                        w[:1],
                                        start_index,
                                        end_index,
                                        device=device
                                    )
                                # ===== Collect TF-MoDISco inputs =====
                                seq_tokens_center = input_ids[0, start_index:end_index].detach().cpu()

                                # token -> attr row ACGT
                                token_to_row = {
                                    6:0,  # A
                                    7:3,  # T
                                    8:1,  # C
                                    9:2   # G
                                }

                                projected_attr = torch.zeros_like(attr)

                                L = seq_tokens_center.shape[0]

                                for pos in range(L):
                                    tok = int(seq_tokens_center[pos])
                                    if tok in token_to_row:
                                        row = token_to_row[tok]
                                        projected_attr[row, pos] = attr[row, pos]

                                attr = projected_attr

                                # ===== Combined figure: ISM (top) + SM logo (bottom), aligned x visually =====
                                fig = plt.figure(figsize=(24,8))

                                # --- ISM heatmap (top) ---
                                ax1 = plt.subplot2grid((2,1), (1,0))
                                sns.heatmap(
                                    ism_map.numpy(),
                                    cmap="coolwarm",
                                    center=0,
                                    yticklabels=["A","T","C","G"],
                                    ax=ax1,
                                    cbar=False
                                )
                                ax1.set_xlabel("Position", fontsize = 18)
                                ax1.set_ylabel("Mutation", fontsize = 18)
                                ax1.set_title("Saturation Mutagenesis Importance", fontsize = 24)
                                # ax1.tick_params(labelsize = 16)
                                # --- small colorbar (top-right corner) ---
                                cbar_ax = fig.add_axes([0.75, 0.88, 0.18, 0.02], zorder=10)
                                norm = plt.Normalize(vmin=ism_map.min().item(), vmax=ism_map.max().item())
                                sm = plt.cm.ScalarMappable(cmap="coolwarm",norm=norm)
                                sm.set_array([])
                                fig.colorbar(sm, cax=cbar_ax,orientation='horizontal')
                                cbar_ax.tick_params(labelsize = 16)

                                # --- SM logo (bottom) ---
                                ax2 = plt.subplot2grid((2,1), (0,0))
                                plot_logo(attr, ax=ax2)
                                ax2.set_xlabel("Genomic Coordinate", fontsize = 18)
                                ax2.set_ylabel("Difference in Prediction", fontsize = 18)
                                ax2.set_title("In-silico Saturation Mutagenesis", fontsize = 24)
                                ax2.tick_params(labelsize = 16)

                                # ===== Force visual alignment without changing internal coordinates =====
                                xmin = 0
                                xmax = ism_map.shape[1]
                                ax1.set_xlim(xmin, xmax)
                                ax2.set_xlim(xmin, xmax)

                                plt.tight_layout()
                                plt.savefig(f"/home/hjzhang/ntv3G4/ISM2/{name}.{conut}.ISM_SM_aligned.pdf")
                                plt.close()
                            else:
                                model.plot_attention(attn, f"{name}attention_map.png")
                    val_preds.extend(pred_labels)
                    val_targets.extend(y.cpu().numpy())
                    val_probs.extend(probs)
                    val_correct += (pred_labels == y.cpu().numpy()).sum()
                    val_total += y.size(0)
                    conut = conut+1
        

Loading existing split indices from L1k_1_k_A549_split_indices.pt


Some weights of the model checkpoint at /home/hjzhang/ntv3G4/nucleotide_transformer_v3/NTv3_100m_post were not used when initializing NTv3PostTrained: ['core.bed_head.head.bias', 'core.bed_head.head.weight', 'core.bed_head.layer_norm.bias', 'core.bed_head.layer_norm.weight', 'core.bigwig_head.species_heads.13.head.bias', 'core.bigwig_head.species_heads.13.head.weight', 'core.bigwig_head.species_heads.13.layer_norm.bias', 'core.bigwig_head.species_heads.13.layer_norm.weight', 'core.bigwig_head.species_heads.16.head.bias', 'core.bigwig_head.species_heads.16.head.weight', 'core.bigwig_head.species_heads.16.layer_norm.bias', 'core.bigwig_head.species_heads.16.layer_norm.weight', 'core.bigwig_head.species_heads.18.head.bias', 'core.bigwig_head.species_heads.18.head.weight', 'core.bigwig_head.species_heads.18.layer_norm.bias', 'core.bigwig_head.species_heads.18.layer_norm.weight', 'core.bigwig_head.species_heads.21.head.bias', 'core.bigwig_head.species_heads.21.head.weight', 'core.bigwig_hea

39572


[Test]:   0%|          | 0/39572 [00:00<?, ?it/s]


NameError: name 'X' is not defined